# Can we predict next-month arrears?

**Question:** from information available at the end of a billing month, can we predict `arrears_next_month`?

Plan: inspect the data → check for leakage and drift → build a time-ordered, account-aware evaluation → compare simple and flexible models → report honestly.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 140)
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(pd.read_csv('columns.csv').to_string(index=False))
print(df.shape, '| accounts:', df.meter_ref.nunique(), '| months:', df.billing_month.min().date(), '->', df.billing_month.max().date())
print('missing values:', int(df.isna().sum().sum()), '| duplicate account-months:', int(df.duplicated(['meter_ref','billing_month']).sum()))
df.describe().T

## 1. Target and basic structure

In [ ]:
print('Overall arrears rate: %.3f' % df.arrears_next_month.mean())
for c in ['late_payments_12m','tariff','occupants']:
    print(df.groupby(c).arrears_next_month.agg(['mean','size']).round(3), '\n')
print('rows per account:'); print(df.groupby('meter_ref').size().describe().round(1))

## 2. Two problems to deal with before modelling

**a) `final_notice_sent` is leakage.** The dictionary says it is sent *in the billing cycle that followed* — i.e. after the billing month, around the time of the outcome we predict. It is almost certainly a consequence of arrears, not a cause, and would not be available when making the prediction.

In [ ]:
print(pd.crosstab(df.final_notice_sent, df.arrears_next_month))
print('AUC of final_notice_sent alone: %.3f' % roc_auc_score(df.arrears_next_month, df.final_notice_sent))

It alone gives a very high AUC, which is what a post-outcome variable looks like. **It is excluded from the real models** (shown once below only to quantify how misleading it would be).

**b) The arrears rate is not stable over time, and accounts repeat.**

In [ ]:
m = df.groupby('billing_month').agg(rate=('arrears_next_month','mean'), n=('meter_ref','size'))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
m.rate.plot(ax=ax[0], marker='o', ms=3); ax[0].set_title('Arrears rate by billing month'); ax[0].set_ylabel('share in arrears')
m.n.plot(ax=ax[1], color='gray'); ax[1].set_title('Rows (accounts) per month')
plt.tight_layout(); plt.show()
print(df.groupby('billing_month').late_payments_12m.mean().iloc[[0,6,12,18,24,-1]].round(3).to_string())

The arrears rate sits around 2-4% until early 2025 and then climbs sharply to roughly a quarter by December 2025. A random row-level split would mix months and put the same account in train and test, which is optimistic. So I evaluate **forward in time** and bootstrap by account.

## 3. Features (only information known at the end of the billing month)
Raw: tariff, occupants, kWh, bill, late payments in 12m. Derived: price per kWh, bill per occupant, kWh vs the account's own previous month, change in late-payment count vs previous month, calendar month. Deliberately **no** absolute time trend (it cannot extrapolate).

In [ ]:
d = df.sort_values(['meter_ref','billing_month']).copy()
g = d.groupby('meter_ref')
prev_ok = (d.billing_month - g.billing_month.shift(1)).dt.days.between(28, 31)   # only truly consecutive months
d['price_per_kwh'] = d.bill_eur / d.kwh
d['bill_per_occ'] = d.bill_eur / d.occupants
d['kwh_chg'] = np.where(prev_ok, d.kwh / g.kwh.shift(1) - 1, np.nan)
d['late_chg'] = np.where(prev_ok, d.late_payments_12m - g.late_payments_12m.shift(1), np.nan)
d['cal_month'] = d.billing_month.dt.month
d = d.sort_index()
NUM = ['occupants','kwh','bill_eur','price_per_kwh','bill_per_occ','late_payments_12m','kwh_chg','late_chg']
CAT = ['tariff']
GB_COLS = NUM + CAT + ['cal_month']
d[NUM].describe().T[['count','mean','std','min','max']].round(3)

## 4. Evaluation design
- **Train:** billing months up to 2025-06. **Test:** 2025-07 to 2025-12 (most recent, highest-arrears period — a hard, realistic case).
- Metrics: ROC AUC (ranking), PR AUC (compare with base rate), Brier score (probability quality).
- 95% CIs: bootstrap over **accounts**, since rows of one account are correlated.
- Baselines: training base rate, and a model using only `late_payments_12m`.

In [ ]:
cut = pd.Timestamp('2025-07-01')
tr, te = d[d.billing_month < cut], d[d.billing_month >= cut]
ytr, yte = tr.arrears_next_month.values, te.arrears_next_month.values
print('train rows %d (rate %.3f) | test rows %d (rate %.3f) | test rows whose account is in train: %.0f%%' %
      (len(tr), ytr.mean(), len(te), yte.mean(), 100 * te.meter_ref.isin(tr.meter_ref).mean()))

def lin(num, cat):
    pre = ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
                             ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
    return make_pipeline(pre, LogisticRegression(max_iter=1000))
def gbm():
    pre = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), CAT)], remainder='passthrough')
    return make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                            l2_regularization=1.0, random_state=0))
models = {
    'Logistic: late_payments_12m only': (lin(['late_payments_12m'], []), ['late_payments_12m']),
    'Logistic: all features':           (lin(NUM, CAT), NUM + CAT),
    'Gradient boosting: all features':  (gbm(), GB_COLS),
}
preds = {'Base rate (train)': np.full(len(te), ytr.mean())}
for name, (mdl, cols) in models.items():
    mdl.fit(tr[cols], ytr); preds[name] = mdl.predict_proba(te[cols])[:, 1]

acc = te.meter_ref.values; ua = np.unique(acc); idx = {a: np.where(acc == a)[0] for a in ua}
rng = np.random.default_rng(0)
boots = [np.concatenate([idx[a] for a in rng.choice(ua, len(ua))]) for _ in range(200)]
metrics = {'ROC AUC': roc_auc_score, 'PR AUC': average_precision_score, 'Brier': brier_score_loss}
rows = []
for name, p in preds.items():
    r = {'model': name}
    for lab, f in metrics.items():
        if name == 'Base rate (train)' and lab != 'Brier':
            r[lab] = '%.3f (by definition)' % (0.5 if lab == 'ROC AUC' else yte.mean()); continue
        bs = [f(yte[b], p[b]) for b in boots]
        r[lab] = '%.3f [%.3f, %.3f]' % (f(yte, p), *np.percentile(bs, [2.5, 97.5]))
    rows.append(r)
pd.DataFrame(rows).set_index('model')

Reference only — what the leaky `final_notice_sent` would appear to deliver (do **not** use):

In [ ]:
lcols = NUM + ['final_notice_sent']
leak = HistGradientBoostingClassifier(max_depth=3, random_state=0).fit(tr[lcols], ytr)
pl = leak.predict_proba(te[lcols])[:, 1]
print('GBM WITH final_notice_sent: ROC AUC %.3f, PR AUC %.3f  (not achievable in practice)' % (roc_auc_score(yte, pl), average_precision_score(yte, pl)))

## 5. Calibration under drift, and what drives the prediction

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in list(preds)[1:]:
    fp, mp = calibration_curve(yte, preds[name], n_bins=8, strategy='quantile'); ax[0].plot(mp, fp, marker='o', label=name)
    pr, rc, _ = precision_recall_curve(yte, preds[name]); ax[1].plot(rc, pr, label=name)
ax[0].plot([0, .6], [0, .6], 'k--', lw=1); ax[0].set_xlabel('predicted'); ax[0].set_ylabel('observed'); ax[0].set_title('Calibration, test period'); ax[0].legend(fontsize=7)
ax[1].axhline(yte.mean(), color='k', ls='--', lw=1); ax[1].set_xlabel('recall'); ax[1].set_ylabel('precision'); ax[1].set_title('Precision-recall (dashed = base rate)')
plt.tight_layout(); plt.show()
print('mean predicted per model:', {k: round(float(v.mean()), 3) for k, v in preds.items()}, '| observed test rate:', round(float(yte.mean()), 3))

In [ ]:
mdl, cols = models['Gradient boosting: all features']
pi = permutation_importance(mdl, te[cols], yte, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=cols).sort_values(ascending=False).round(4)

## 6. Robustness: rolling-origin check
Retrain on all earlier months and score each of the last 6 months separately.

In [ ]:
rows = []
for m_ in pd.date_range('2025-07-01', '2025-12-01', freq='MS'):
    a, b = d[d.billing_month < m_], d[d.billing_month == m_]
    p = gbm().fit(a[GB_COLS], a.arrears_next_month).predict_proba(b[GB_COLS])[:, 1]
    yb = b.arrears_next_month
    rows.append({'month': m_.strftime('%Y-%m'), 'observed rate': yb.mean(), 'mean predicted': p.mean(),
                 'GBM ROC AUC': roc_auc_score(yb, p), 'late_payments-only ROC AUC': roc_auc_score(yb, b.late_payments_12m),
                 'GBM PR AUC': average_precision_score(yb, p)})
pd.DataFrame(rows).round(3)

## 7. Conclusion

*(Figures below are from the run at time of writing; re-run the notebook to refresh them.)*

**Yes, but only moderately — and only if `final_notice_sent` is left out.**

- **Signal exists.** On the held-out Jul–Dec 2025 period, logistic regression on features known at month end reaches ROC AUC ≈ 0.74 (95% CI ≈ 0.72–0.75) and PR AUC ≈ 0.50 against a 0.22 base rate — roughly 2.3× better than chance on precision. Gradient boosting does no better than logistic regression, so there is no evidence of useful non-linear structure.
- **Main driver: past late payments.** Arrears rate climbs steeply with `late_payments_12m` (in the test period ≈ 13% at 0, 33% at 1, 61% at 2, 85% at 3+). Bill size is the next signal (arrears ≈ 10% in the lowest bill quintile vs ≈ 32% in the highest). Tariff, occupants and the month-to-month changes add essentially nothing.
- **Leakage.** `final_notice_sent` yields AUC ≈ 0.96, but it is sent *after* the billing month and tracks the outcome (87% vs 1.4% arrears rate); it is not available at prediction time. Using it would make the model look excellent and be useless in practice.
- **Drift is the big caveat.** The arrears rate rose from ≈ 3% to ≈ 26% during 2025 while the late-payment mix stayed flat, so something outside these columns changed. Models trained on earlier months are badly **under-calibrated** on the test period (gradient boosting predicts ≈ 4–7% vs 15–26% observed). Ranking (AUC) holds up month by month; absolute probabilities do not. Use the model to *rank/prioritise* accounts, or recalibrate on recent data, rather than reading its probabilities literally.

**Limitations / next steps:** one train/test split on a single shifting regime; no information on *why* the rate rose (price change, policy, collections practice?) — worth asking the business; only ~2 features carry signal, so payment history (days late, partial payments, balance owed) would likely help far more than anything in this table. If the aim is operational (e.g. who to contact), choose a threshold from the PR curve using the cost of outreach vs. missed arrears.